In [ ]:
import os
import json
import pandas as pd
import pyarrow.ipc as ipc
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

RAW_DIR = Path(r'C:\Users\nirmi\Desktop\Capstone\data\raw')
OUT_DIR = RAW_DIR / 'health11k_text'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load the IDs we need ──────────────────────────────────────────────────
with open(RAW_DIR / 'health11k/train/data-00000-of-00001.arrow', 'rb') as f:
    meta_df = ipc.open_stream(f).read_all().to_pandas()

wildchat_ids = set(meta_df[meta_df['dataset_source'] == 'wildchat']['conversation_id'])
lmsys_ids    = set(meta_df[meta_df['dataset_source'] == 'lmsys']['conversation_id'])

print(f'WildChat IDs needed : {len(wildchat_ids):,}')
print(f'LMSYS IDs needed    : {len(lmsys_ids):,}')

In [ ]:
# ── 1. WildChat (public) ──────────────────────────────────────────────────
wc_out = OUT_DIR / 'wildchat_conversations.parquet'

if wc_out.exists():
    wc_df = pd.read_parquet(wc_out)
    print(f'WildChat already downloaded: {wc_df["conversation_id"].nunique():,} conversations — skipping.')
else:
    print('Streaming allenai/WildChat  (may take 15-30 min) ...')
    records, remaining = [], wildchat_ids.copy()
    pbar = tqdm(total=len(remaining), desc='WildChat matched')

    for row in load_dataset('allenai/WildChat', split='train', streaming=True):
        cid = row.get('conversation_id')
        if cid in remaining:
            for i, turn in enumerate(row.get('conversation', [])):
                records.append({
                    'conversation_id': cid,
                    'turn_index': i,
                    'role': turn.get('role', ''),
                    'content': turn.get('content', ''),
                    'language': row.get('language', ''),
                    'model': row.get('model', ''),
                })
            remaining.discard(cid)
            pbar.update(1)
            if not remaining:
                break
    pbar.close()

    wc_df = pd.DataFrame(records)
    wc_df.to_parquet(wc_out, index=False)
    print(f'Saved {wc_df["conversation_id"].nunique():,} / {len(wildchat_ids):,} WildChat conversations  →  {wc_out}')
    if remaining:
        print(f'  {len(remaining):,} IDs not found in WildChat')

In [ ]:
# ── 2. LMSYS (gated) ─────────────────────────────────────────────────────
lm_out = OUT_DIR / 'lmsys_conversations.parquet'

if lm_out.exists():
    lm_df = pd.read_parquet(lm_out)
    print(f'LMSYS already downloaded: {lm_df["conversation_id"].nunique():,} conversations — skipping.')
else:
    print('Streaming lmsys/lmsys-chat-1m  (may take 45-90 min) ...')
    try:
        records, remaining = [], lmsys_ids.copy()
        pbar = tqdm(total=len(remaining), desc='LMSYS matched')

        for row in load_dataset('lmsys/lmsys-chat-1m', split='train', streaming=True):
            cid = row.get('conversation_id')
            if cid in remaining:
                for i, turn in enumerate(row.get('conversation', [])):
                    records.append({
                        'conversation_id': cid,
                        'turn_index': i,
                        'role': turn.get('role', ''),
                        'content': turn.get('content', ''),
                        'language': row.get('language', ''),
                        'model': row.get('model', ''),
                    })
                remaining.discard(cid)
                pbar.update(1)
                if not remaining:
                    break
        pbar.close()

        lm_df = pd.DataFrame(records)
        lm_df.to_parquet(lm_out, index=False)
        print(f'Saved {lm_df["conversation_id"].nunique():,} / {len(lmsys_ids):,} LMSYS conversations  →  {lm_out}')
        if remaining:
            print(f'  {len(remaining):,} IDs not found in LMSYS')

    except Exception as e:
        if 'gated' in str(e).lower() or 'access' in str(e).lower():
            print('ACCESS DENIED for lmsys/lmsys-chat-1m.')
            print('Go to https://huggingface.co/datasets/lmsys/lmsys-chat-1m')
            print('Click "Agree and access repository", then re-run this cell.')
        else:
            raise

In [ ]:
# ── Report ────────────────────────────────────────────────────────────────
report = {}
for name, path, needed in [('wildchat', wc_out, wildchat_ids), ('lmsys', lm_out, lmsys_ids)]:
    if path.exists():
        df = pd.read_parquet(path)
        got = df['conversation_id'].nunique()
        report[name] = {'needed': len(needed), 'downloaded': got,
                         'turns': len(df), 'status': 'complete' if got == len(needed) else 'partial'}
    else:
        report[name] = {'needed': len(needed), 'downloaded': 0, 'status': 'missing'}

with open(OUT_DIR / 'download_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
total = sum(v['downloaded'] for v in report.values())
needed = len(wildchat_ids) + len(lmsys_ids)
print(f'\nTotal conversations with text: {total:,} / {needed:,} ({100*total/needed:.1f}%)')
print('\nNext step: run notebooks/07_rebuild_health11k.ipynb')